# RoutIR Client Tutorial

This notebook walks through `routir.client`, the gRPC-default / REST-fallback client for a RoutIR search service. We assume you have a RoutIR server running locally or remotely; set the endpoints in the next cell. The notebook is async-first (Jupyter 7+ allows top-level `await`), with a parallel sync example for users who would rather not deal with `asyncio` at all.

In [ ]:
from routir.client import AsyncClient, Client, RoutirClientError

# Edit these to point at your server.
REST_ENDPOINT = "http://localhost:5000"
GRPC_ENDPOINT = "localhost:50051"   # set to None if your server was started without --grpc
API_KEY = None                      # set to your token if the server was started with --api_key

## Two ways to use the client

- `AsyncClient` is the primary API. It is async, batches naturally with the rest of an   `asyncio` program, and is what every other Python component in RoutIR ultimately talks to.
- `Client` is a thin synchronous wrapper. It owns a private event loop on a background thread,   so you can call it from a normal script (or a notebook cell that you would rather not mark   `await`-ful) without juggling `asyncio.run`.

Both expose the same methods (`search`, `score`, `content`, `pipeline`, `avail`, `ping`, `search_batch`, `score_batch`). The dicts going in and out are identical.

Here is the minimal sync version. `ping` returns a status string; `avail` returns the services registered on the server, grouped by role (`search`, `score`, `content`, `fuse`, `decompose_query`), plus any pipeline aliases the server has configured. Use `Client` as a context manager so the background loop is torn down cleanly.

In [ ]:
with Client(
    endpoint=REST_ENDPOINT,
    grpc_endpoint=GRPC_ENDPOINT,
    api_key=API_KEY,
) as c:
    print("ping:", c.ping())
    avail = c.avail()
    print("transport:", c.transport)
    print("roles:", list(avail.keys()))
    # avail['search'] is a list of service names; trim to keep output small.
    print("search services (first 5):", avail.get("search", [])[:5])

## Transport selection

The `transport` argument decides how the client talks to the server:

- `transport="auto"` (the default) probes gRPC with a single `Ping` on the first call.   If the probe succeeds, the client commits to gRPC for the rest of its life. If the probe   fails for a transport-level reason (no `grpcio` installed, channel error, `UNIMPLEMENTED`),   the client logs a warning and silently falls back to REST.
- `transport="grpc"` forces gRPC. A failed probe raises `RoutirClientError` rather than falling   back.
- `transport="rest"` skips gRPC entirely; useful when you know the server has no gRPC port.

Auth failures (`UNAUTHENTICATED`, `PERMISSION_DENIED`) are *not* treated as transport failures. Wrong credentials should not silently downgrade you to a different wire format, so the client raises in that case.

## Async usage

Top-level `await` works in Jupyter 7 / IPython 8 and later. Outside a notebook, wrap the block in `asyncio.run(main())`.

In [ ]:
async with AsyncClient(
    endpoint=REST_ENDPOINT,
    grpc_endpoint=GRPC_ENDPOINT,
    api_key=API_KEY,
) as c:
    print("ping:", await c.ping())
    avail = await c.avail()
    print("transport selected:", c.transport)
    print("pipeline aliases:", avail.get("pipeline_aliases", {}))

## Search and score

`search(service, query, **kwargs)` calls a retrieval service. Common kwargs: `limit` (top-k), `subset` (restrict to a list of doc IDs), `instruction` (model-specific prompt prefix). The return value is a dict:

```python
{"query": "...", "scores": {docid: score, ...}, "service": "...", "cached": bool, "timestamp": float}
```

`score(service, query, passages, **kwargs)` calls a pointwise scorer / reranker over a list of passage strings. Its return value has `scores` as a *list of floats* (parallel to the input), not a dict.

Adapt `SEARCH_SERVICE` and `SCORE_SERVICE` below to names that exist on your server (check `avail` output above).

In [ ]:
SEARCH_SERVICE = "YOUR-SEARCH-SERVICE"   # e.g. "bm25-msmarco"
SCORE_SERVICE = "YOUR-SCORE-SERVICE"     # e.g. "llm-scorer"

async with AsyncClient(
    endpoint=REST_ENDPOINT,
    grpc_endpoint=GRPC_ENDPOINT,
    api_key=API_KEY,
) as c:
    try:
        s = await c.search(SEARCH_SERVICE, query="what is photosynthesis", limit=5)
        print("search hits:", len(s["scores"]))
        print("top doc id:", next(iter(s["scores"])))
    except RoutirClientError as e:
        print("search failed (likely service not registered):", e)

    try:
        r = await c.score(
            SCORE_SERVICE,
            query="what is photosynthesis",
            passages=[
                "Photosynthesis is the process by which plants convert light into chemical energy.",
                "The Eiffel Tower is in Paris.",
            ],
        )
        print("score result:", r["scores"])
    except RoutirClientError as e:
        print("score failed:", e)

## Pipelines

RoutIR ships a tiny DSL for composing services. A pipeline is a string parsed by the server:

- `bm25%100` runs the `bm25` service and keeps the top 100 documents.
- `bm25%100 >> rerank%20` runs `bm25` then reranks the top 100 with `rerank`, keeping the top 20.
- `{dense, sparse}RRF%100` runs `dense` and `sparse` in parallel and fuses with RRF.
- `bm25%100[ret] >> rerank[rr]%20` aliases stages so you can target them by name in `runtime_kwargs`.

See `routir.pipeline.parser` for the full grammar. A reranking stage needs a `collection` so the server can fetch document content; for retrieve-only pipelines you can omit it.

In [ ]:
# Adapt to your services / collection. The pipeline below assumes:
#   - a search service called `bm25`
#   - a reranker called `rerank` aliased as `rr`
#   - a collection called `msmarco-passage` with a content endpoint
PIPELINE = "bm25%100 >> rerank[rr]%20"
COLLECTION = "msmarco-passage"

async with AsyncClient(
    endpoint=REST_ENDPOINT,
    grpc_endpoint=GRPC_ENDPOINT,
    api_key=API_KEY,
) as c:
    try:
        out = await c.pipeline(
            pipeline=PIPELINE,
            query="how do solar panels work",
            collection=COLLECTION,
            # runtime_kwargs target a stage by alias and override its kwargs at call time.
            runtime_kwargs={"rr": {"prompt": "Is this passage relevant to the query?"}},
        )
        print("pipeline hits:", len(out["scores"]))
        print("cached:", out["cached"])
    except RoutirClientError as e:
        print("pipeline failed:", e)

## Batch APIs

`search_batch(payloads)` and `score_batch(payloads)` accept a list of fully-formed payload dicts. Each payload is independent (different query, `limit`, `subset`, etc.), and the client issues them concurrently. This is the preferred shape for fanning out many queries against the same service, since the server can batch them on its side.

In [ ]:
queries = [
    "how do solar panels work",
    "what is photosynthesis",
    "who wrote the iliad",
]
payloads = [{"service": SEARCH_SERVICE, "query": q, "limit": 10} for q in queries]

async with AsyncClient(
    endpoint=REST_ENDPOINT,
    grpc_endpoint=GRPC_ENDPOINT,
    api_key=API_KEY,
) as c:
    try:
        results = await c.search_batch(payloads)
        for q, r in zip(queries, results):
            print(f"{q!r}: {len(r['scores'])} hits")
    except RoutirClientError as e:
        print("batch search failed:", e)

## Error handling

Every unrecoverable failure surfaces as `RoutirClientError`. The client retries transient errors automatically before raising:

- REST: connection errors and HTTP 5xx are retried with exponential backoff (0.1s, 0.2s, 0.4s,   capped at 2s; total attempts = `retries + 1`). HTTP 4xx raises immediately.
- gRPC: `UNAVAILABLE`, `DEADLINE_EXCEEDED`, `RESOURCE_EXHAUSTED`, `ABORTED` retry.   `INVALID_ARGUMENT`, `NOT_FOUND`, `UNAUTHENTICATED`, `PERMISSION_DENIED`, `UNIMPLEMENTED`, and   `FAILED_PRECONDITION` raise immediately.

Every `RoutirClientError` from `GrpcTransport` includes the status-code name in its message (e.g. `gRPC UNAUTHENTICATED: ...`), which is also how the auto-fallback logic decides whether to downgrade to REST.

In [ ]:
async with AsyncClient(
    endpoint=REST_ENDPOINT,
    grpc_endpoint=GRPC_ENDPOINT,
    api_key=API_KEY,
) as c:
    try:
        await c.search("definitely-not-a-real-service", query="test")
    except RoutirClientError as e:
        print("caught RoutirClientError:")
        print(" ", e)

## TLS and nginx fronts

The default deployment puts an nginx (or other reverse proxy) in front of RoutIR and terminates TLS there, so the client talks plaintext HTTP/h2c to the proxy on localhost or a private network. That is what every cell above assumes.

For a directly TLS-fronted server, two options:

- REST: use an `https://` endpoint. `aiohttp` handles the rest.
- gRPC: either prefix the gRPC endpoint with `grpcs://` (the client strips the scheme and flips   to secure-channel mode) or pass `tls=True` explicitly.

If you set `tls=True` and use a `grpcs://`-prefixed endpoint, the explicit flag wins; the scheme is only an *inference* hint when `tls=None`. The cell below is illustrative; the `if False:` guard prevents it from connecting.

In [ ]:
# Non-executable example — do not run unless you have a TLS-fronted server.
if False:
    async with AsyncClient(
        endpoint="https://routir.example.com",
        grpc_endpoint="grpcs://routir.example.com:443",
        api_key="sk-...",
        # tls=True,   # only needed if grpc_endpoint has no grpcs:// prefix
    ) as c:
        print(await c.ping())

# Two more things worth knowing:
# - `await client.reset_transport()` closes the current transport and clears the cached choice;
#   the next call re-probes. Useful if the server cycled from REST-only to gRPC-capable while
#   the client was alive.
# - `routir.client` has a strict dependency floor (stdlib + aiohttp + optional grpcio) and
#   does not import any of routir.models / routir.processors / routir.config, so it is safe to
#   pull into an inference-only environment without dragging in PyTorch. See src/routir/client/
#   for the full source; the wire-format-agnostic dict shapes live in
#   routir.client.transport.Transport.